In [1]:
# --- Install & Import PySpark ---
!apt-get install openjdk-11-jdk-headless -qq > /dev/null
!pip install pyspark -q

import pyspark
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
from pyspark.sql import functions as F

In [2]:
# --- Create SparkSession ---
spark = SparkSession.builder \
    .appName("Spark SQL Joins and Schema Example") \
    .config("spark.ui.showConsoleProgress", "false") \
    .getOrCreate()

print("Spark version:", spark.version)

Spark version: 3.5.1


In [5]:
# --- Define Schema using StructType ---
employee_schema = StructType([
    StructField("emp_id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("dept_id", IntegerType(), True)
])
department_schema = StructType([
    StructField("dept_id", IntegerType(), True),
    StructField("dept_name", StringType(), True)
])

In [7]:
# --- Sample Data ---
employee_data = [
    (1, "Gayatri", 101),
    (2, "Babali", 102),
    (3, "Charlie", None),
    (4, "Shubhangi", 103)
]

department_data = [
    (101, "HR"),
    (102, "Finance"),
    (104, "IT")
]

In [8]:
# --- Create DataFrames with Schema ---
df_emp = spark.createDataFrame(employee_data, schema=employee_schema)
df_dept = spark.createDataFrame(department_data, schema=department_schema)

print("Employees DataFrame:")
df_emp.show()

print("Departments DataFrame:")
df_dept.show()

Employees DataFrame:
+------+---------+-------+
|emp_id|     name|dept_id|
+------+---------+-------+
|     1|  Gayatri|    101|
|     2|   Babali|    102|
|     3|  Charlie|   NULL|
|     4|Shubhangi|    103|
+------+---------+-------+

Departments DataFrame:
+-------+---------+
|dept_id|dept_name|
+-------+---------+
|    101|       HR|
|    102|  Finance|
|    104|       IT|
+-------+---------+



In [9]:

# --- Register DataFrames as Temp Views for SQL ---
df_emp.createOrReplaceTempView("employees")
df_dept.createOrReplaceTempView("departments")


In [10]:
# --- Spark SQL Joins ---
print("Inner Join:")
spark.sql("""
SELECT e.emp_id, e.name, d.dept_name
FROM employees e
INNER JOIN departments d
ON e.dept_id = d.dept_id
""").show()

print("Left Join:")
spark.sql("""
SELECT e.emp_id, e.name, d.dept_name
FROM employees e
LEFT JOIN departments d
ON e.dept_id = d.dept_id
""").show()

print("Right Join:")
spark.sql("""
SELECT e.emp_id, e.name, d.dept_name
FROM employees e
RIGHT JOIN departments d
ON e.dept_id = d.dept_id
""").show()

print("Full Outer Join:")
spark.sql("""
SELECT e.emp_id, e.name, d.dept_name
FROM employees e
FULL OUTER JOIN departments d
ON e.dept_id = d.dept_id
""").show()


Inner Join:
+------+-------+---------+
|emp_id|   name|dept_name|
+------+-------+---------+
|     1|Gayatri|       HR|
|     2| Babali|  Finance|
+------+-------+---------+

Left Join:
+------+---------+---------+
|emp_id|     name|dept_name|
+------+---------+---------+
|     1|  Gayatri|       HR|
|     2|   Babali|  Finance|
|     4|Shubhangi|     NULL|
|     3|  Charlie|     NULL|
+------+---------+---------+

Right Join:
+------+-------+---------+
|emp_id|   name|dept_name|
+------+-------+---------+
|     1|Gayatri|       HR|
|     2| Babali|  Finance|
|  NULL|   NULL|       IT|
+------+-------+---------+

Full Outer Join:
+------+---------+---------+
|emp_id|     name|dept_name|
+------+---------+---------+
|     3|  Charlie|     NULL|
|     1|  Gayatri|       HR|
|     2|   Babali|  Finance|
|     4|Shubhangi|     NULL|
|  NULL|     NULL|       IT|
+------+---------+---------+



In [11]:
# --- Using Spark SQL Functions ---
df_emp_with_newcol = df_emp.withColumn(
    "name_upper", F.upper(F.col("name"))
).withColumn(
    "is_dept_assigned", F.when(F.col("dept_id").isNotNull(), F.lit("Yes")).otherwise(F.lit("No"))
).withColumn(
    "emp_info", F.concat(F.col("name"), F.lit(" (ID: "), F.col("emp_id"), F.lit(")"))
)

print("DataFrame with SQL Functions Applied:")
df_emp_with_newcol.show(truncate=False)


DataFrame with SQL Functions Applied:
+------+---------+-------+----------+----------------+-----------------+
|emp_id|name     |dept_id|name_upper|is_dept_assigned|emp_info         |
+------+---------+-------+----------+----------------+-----------------+
|1     |Gayatri  |101    |GAYATRI   |Yes             |Gayatri (ID: 1)  |
|2     |Babali   |102    |BABALI    |Yes             |Babali (ID: 2)   |
|3     |Charlie  |NULL   |CHARLIE   |No              |Charlie (ID: 3)  |
|4     |Shubhangi|103    |SHUBHANGI |Yes             |Shubhangi (ID: 4)|
+------+---------+-------+----------+----------------+-----------------+



In [12]:
# --- Example of filtering using SQL Functions ---
print("Employees without Department:")
df_emp.filter(F.col("dept_id").isNull()).show()


Employees without Department:
+------+-------+-------+
|emp_id|   name|dept_id|
+------+-------+-------+
|     3|Charlie|   NULL|
+------+-------+-------+

